# DEH 30-Day PySpark Challenge
## Day 16 — Window Functions: row_number, rank, dense_rank

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name",  StringType(), True),
    StructField("last_name",   StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("state",       StringType(), True),
    StructField("country",     StringType(), True),
    StructField("signup_date", DateType(),   True),
    StructField("segment",     StringType(), True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()}")

## Step 5 — Define a WindowSpec

In [ ]:
# WindowSpec — partition by region, order by unit_price descending
window = Window.partitionBy("region").orderBy(F.col("unit_price").desc())
print("WindowSpec defined")
print("partitionBy: region")
print("orderBy: unit_price descending")

## Step 6 — row_number()

In [ ]:
# Assign unique row numbers within each region
orders_df.withColumn("row_num", F.row_number().over(window)) \
    .select("region", "order_id", "unit_price", "row_num") \
    .orderBy("region", "row_num") \
    .show(10)

## Step 7 — Top 1 per group using row_number()

In [ ]:
# Most expensive order per region
orders_df.withColumn("row_num", F.row_number().over(window)) \
    .filter(F.col("row_num") == 1) \
    .select("region", "order_id", "unit_price") \
    .orderBy("region") \
    .show()

## Step 8 — rank() vs dense_rank() vs row_number()

In [ ]:
# Compare all three — observe how ties are handled
orders_df.withColumn("row_num",    F.row_number().over(window)) \
         .withColumn("rank",       F.rank().over(window)) \
         .withColumn("dense_rank", F.dense_rank().over(window)) \
    .select("region", "order_id", "unit_price", "row_num", "rank", "dense_rank") \
    .filter(F.col("region") == "East") \
    .orderBy("rank") \
    .show(10)

## Step 9 — Top 3 per region using dense_rank()

In [ ]:
top3_per_region = orders_df \
    .withColumn("rank", F.dense_rank().over(window)) \
    .filter(F.col("rank") <= 3) \
    .select("region", "order_id", "customer_id", "unit_price", "rank") \
    .orderBy("region", "rank")

top3_per_region.show()

---
## Assignment — Day 16

---

### Task 1
Using `row_number()`, number all orders within each `region` ordered by `unit_price` descending.  
Show `region`, `order_id`, `unit_price`, and `row_num`.

In [ ]:
# Task 1 — Write your code here


### Task 2
Apply all three — `row_number()`, `rank()`, and `dense_rank()` — partitioned by `region`, ordered by `unit_price` descending.  
Show the first 10 rows for the East region and observe where results differ.

In [ ]:
# Task 2 — Write your code here


### Task 3
Using `dense_rank()`, find the top 2 highest value orders per `region`.  
Filter for `rank <= 2` and show the result.

In [ ]:
# Task 3 — Write your code here


### Task 4
For each customer, rank their orders by `unit_price` descending using `row_number()`.  
Filter to keep only each customer's most expensive order (`row_num == 1`).  
Join with `customers.csv` to show the customer's full name.

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*